In [1]:
import pandas as pd
import numpy as np

In [2]:
# constants
total_customers = 1_000_000
vehicle_value = 100000
premium_rate = 100
claim_rate = 0.10

In [3]:
tenure_dist = [1,2,3,4]
tenure_prob = [0.20,0.30,0.40,0.10]

In [4]:
# policy dataset
dates = pd.date_range("2024-01-01","2024-12-31")
base = total_customers // len(dates)
extra = total_customers % len(dates)

In [5]:
counts = [base + (1 if i < extra else 0) for i in range(len(dates))]
purchase_dates = np.repeat(dates, counts)

tenures = np.random.choice(tenure_dist, total_customers, p=tenure_prob)


In [6]:
policy_df = pd.DataFrame({
    "customer_id":[f"c{i+1:07d}" for i in range(total_customers)],
    "vehicle_id":[f"v{i+1:07d}" for i in range(total_customers)],
    "vehicle_value":vehicle_value,
    "premium":tenures * premium_rate,
    "policy_purchase_date":purchase_dates,
    "policy_tenure":tenures
})


In [7]:
policy_df["policy_start_date"] = pd.to_datetime(policy_df["policy_purchase_date"]) + pd.Timedelta(days=365)
policy_df["policy_end_date"] = policy_df["policy_start_date"] + pd.to_timedelta(policy_df["policy_tenure"]*365, unit="d")


In [8]:
# claims dataset
claim_amount = vehicle_value * claim_rate


In [9]:
mask = policy_df["policy_purchase_date"].dt.day.isin([7,14,21,28])
eligible_25 = policy_df[mask]

In [10]:
claim_25 = eligible_25.sample(round(len(eligible_25)*0.30), random_state=42)
claim_25["claim_date"] = claim_25["policy_start_date"]
claim_25["claim_type"] = 1

In [11]:
eligible_26 = policy_df[policy_df["policy_tenure"]==4]
claim_26 = eligible_26.sample(round(len(eligible_26)*0.10), random_state=7)

In [12]:
period = pd.date_range("2026-01-01","2026-02-28")
claim_26["claim_date"] = np.resize(period.values, len(claim_26))

In [13]:

claimed_25 = set(claim_25["vehicle_id"])
claim_26["claim_type"] = claim_26["vehicle_id"].apply(lambda x: 2 if x in claimed_25 else 1)


In [14]:
claims_df = pd.concat([
    claim_25[["customer_id","vehicle_id","claim_date","claim_type"]],
    claim_26[["customer_id","vehicle_id","claim_date","claim_type"]]
]).reset_index(drop=True)

In [15]:
claims_df.insert(0,"claim_id",[f"cl{i+1:07d}" for i in range(len(claims_df))])
claims_df["claim_amount"] = claim_amount

In [16]:
# analysis
total_premium = policy_df["premium"].sum()

In [18]:
claims_df["year"] = claims_df["claim_date"].dt.year
claims_df["month"] = claims_df["claim_date"].dt.month


In [17]:
veh_meta = policy_df.set_index("vehicle_id")[["policy_tenure","premium","policy_purchase_date"]]
claims_df = claims_df.join(veh_meta,on="vehicle_id")

In [19]:
claim_cost = claims_df.groupby("year")["claim_amount"].sum()

In [20]:
prem_by_tenure = policy_df.groupby("policy_tenure")["premium"].sum()
claim_by_tenure = claims_df.groupby("policy_tenure")["claim_amount"].sum()

In [21]:

loss_ratio = claim_by_tenure / prem_by_tenure

In [22]:
print("total premium:", total_premium)
print("claim cost by year:\n", claim_cost)
print("loss ratio by tenure:\n", loss_ratio)

total premium: 239963100
claim cost by year:
 year
2025    393440000.0
2026     99850000.0
Name: claim_amount, dtype: float64
loss ratio by tenure:
 policy_tenure
1    3.927356
2    1.956027
3    1.318020
4    3.484331
dtype: float64
